In [3]:
import pandas as pd
from scipy.fft import fft
import numpy as np

In [ ]:
from scipy.signal import butter, lfilter

def butter_bandpass(lowcut, highcut, fs, order=5):#bandpass
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):#bandpass filter
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

def signal_frequency_band_energies(sampled_signal, frequency_bands, sampling_frequency,
order=5):#signal frequncy to energies
    energies = []
    for bands in frequency_bands:
        energies.append(np.sum(np.abs(butter_bandpass_filter(sampled_signal, bands[0], bands[1],
        sampling_frequency, order))**2))
    return energies

In [5]:
data = pd.read_csv("./dataSet.csv")#get data from csv
group=data.groupby(['subject','label'])#group by subject and label
print(group.size())#show groups size

subject  label
S10      1        826000
         2        507500
S11      1        826000
         2        476000
S13      1        826001
         2        464800
S14      1        826000
         2        472500
S15      1        822500
         2        480200
S16      1        826000
         2        471101
S17      1        826700
         2        506100
S2       1        800800
         2        430500
S3       1        798000
         2        448000
S4       1        810601
         2        444500
S5       1        838600
         2        451500
S6       1        826000
         2        455000
S7       1        830200
         2        448000
S8       1        818300
         2        469000
S9       1        826000
         2        451500
dtype: int64


In [ ]:
keys=group.groups.keys()#get and show groups keys
print(keys)

dict_keys([('S10', 1), ('S10', 2), ('S11', 1), ('S11', 2), ('S13', 1), ('S13', 2), ('S14', 1), ('S14', 2), ('S15', 1), ('S15', 2), ('S16', 1), ('S16', 2), ('S17', 1), ('S17', 2), ('S2', 1), ('S2', 2), ('S3', 1), ('S3', 2), ('S4', 1), ('S4', 2), ('S5', 1), ('S5', 2), ('S6', 1), ('S6', 2), ('S7', 1), ('S7', 2), ('S8', 1), ('S8', 2), ('S9', 1), ('S9', 2)])


In [7]:
windows_size=64
process_data=[]
for i in keys:
    print(i)
    temp=group.get_group(i)
    temp_feature=temp.drop('subject',axis=1).values.tolist()
    temp_subject=temp['subject'].values.tolist()
    print(len(temp))
    begin=0
    end=windows_size
    while end<len(temp):
        temp_process_data=[]
        temp_data=np.array(temp_feature[begin:end],dtype="float32").T
        for x in range(1,4):#ACC0,ACC1,ACC2
            temp_process_data.append(temp_data[x].mean())
        for x in range(5,9):#EMG,EDA,resp,temp
            temp_process_data.append(temp_data[x].mean())
        #ECG_ULF,ECG_LF,ECG_HF,ECG_UHF
        for x in signal_frequency_band_energies(fft(temp_data[4]),[[0.01, 0.04], [0.04, 0.15], [0.15, 0.4], [0.4, 1.0]],700):
            temp_process_data.append(x)
        temp_process_data.append(temp_data[9][0])#label
        temp_process_data.append(temp_subject[0])#subject
        #print(temp_process_data)
        process_data.append(temp_process_data)
        begin+=windows_size
        end+=windows_size


('S10', 1)
826000
('S10', 2)
507500
('S11', 1)
826000
('S11', 2)
476000
('S13', 1)
826001
('S13', 2)
464800
('S14', 1)
826000
('S14', 2)
472500
('S15', 1)
822500
('S15', 2)
480200
('S16', 1)
826000
('S16', 2)
471101
('S17', 1)
826700
('S17', 2)
506100
('S2', 1)
800800
('S2', 2)
430500
('S3', 1)
798000
('S3', 2)
448000
('S4', 1)
810601
('S4', 2)
444500
('S5', 1)
838600
('S5', 2)
451500
('S6', 1)
826000
('S6', 2)
455000
('S7', 1)
830200
('S7', 2)
448000
('S8', 1)
818300
('S8', 2)
469000
('S9', 1)
826000
('S9', 2)
451500


In [8]:
process_data=pd.DataFrame(process_data)
process_data=process_data.rename(columns={0:"ACC0",
                                          1:"ACC1",
                                          2:"ACC2",
                                          3:"EMG",
                                          4:"EDA",
                                          5:"resp",
                                          6:"temp",
                                          7:"ECG_ULF",
                                          8:"ECG_LF",
                                          9:"ECG_HF",
                                          10:"ECG_UHF",
                                          11:"label",
                                          12:"subject"})
process_data.to_csv("process_dataSet.csv")